# Assignment 1 — training pipeline

You're using a notebook for this stage. If you'd rather use a script,
write `train.py` instead and delete this file — `main.py` refuses to run
if it finds both `train.py` and `train.ipynb`, so exactly one of them must
exist.

Replace this cell and everything below it with your own code. When run via
`uv run python main.py train`, this notebook must, start to finish, with
no manual steps in between:

- Load your committed dataset.
- Apply the feature engineering, preprocessing, and train/val/test split
  consistent with the business framing you wrote in `REPORT.md`.
- Train the same model — linear regression for a regression target,
  logistic regression for a binary one — three ways, on the same split:
  scikit-learn, a from-scratch PyTorch loop (Session 5's manual approach:
  raw tensors, autograd, a manual gradient-step update), and the standard
  `torch.nn.Module` + `torch.optim` workflow.
- Evaluate all three against each other and against a naive baseline.
- Save whatever `app.py`/`app.ipynb` needs to build its dashboard without
  retraining anything.

How you structure the code beyond that — cell layout, function
boundaries, naming — is your call to make and be able to defend.

In [31]:
# --- ALL IMPORTS IN ONE PLACE (so the notebook works from a blank start) ---
import os, json                                       # os = folders and files, json = text settings files
import joblib                                         # saves and loads scikit-learn objects
import numpy as np                                    # maths tools
import pandas as pd                                   # tables
import torch                                          # PyTorch: tensors and automatic gradients
import torch.nn as nn                                 # PyTorch's building blocks for models

from sklearn.model_selection import GroupShuffleSplit    # the split that keeps each host's flats together
from sklearn.preprocessing import StandardScaler         # rescales number columns
from sklearn.linear_model import LinearRegression, Ridge # plain linear regression, and the version with a penalty
from sklearn.metrics import mean_absolute_error, r2_score  # MAE = average miss in euros, R2 = share explained

#### Load and filtering data

In [32]:
import pandas as pd

df = pd.read_csv("data/listings.csv")

# price is text like "$85.00" -> turn it into a number
df["price"] = df["price"].str.replace("[$,]", "", regex=True).astype(float)

# can't train on a blank answer, so drop rows without a price
df = df.dropna(subset=["price"])

# scope: entire homes with exactly 2 bedrooms
df = df[(df["room_type"] == "Entire home/apt") & (df["bedrooms"] == 2)]

df = df[df["price"] <= 500]
print(len(df))

3591


In [33]:
# Check missing values and data types for candidate columns
#Before cleaning, see which candidate columns have missing values or the wrong type
candidates = [
    "host_id", "accommodates", "beds", "bathrooms", "bathrooms_text",
    "amenities", "latitude", "longitude", "neighbourhood_cleansed",
    "neighbourhood_group_cleansed", "property_type", "host_since",
    "host_is_superhost", "host_response_rate", "host_acceptance_rate",
    "host_listings_count", "minimum_nights", "maximum_nights",
    "instant_bookable", "availability_365", "number_of_reviews",
    "review_scores_rating", "review_scores_location", "last_review",
]

print(df[candidates].isna().sum())
print(df[candidates].dtypes)

host_id                            0
accommodates                       0
beds                              31
bathrooms                       3591
bathrooms_text                     0
amenities                          0
latitude                           0
longitude                          0
neighbourhood_cleansed             0
neighbourhood_group_cleansed       0
property_type                      0
host_since                      3591
host_is_superhost                  0
host_response_rate              3591
host_acceptance_rate            3591
host_listings_count                0
minimum_nights                   303
maximum_nights                   303
instant_bookable                3591
availability_365                   0
number_of_reviews                  0
review_scores_rating             420
review_scores_location           420
last_review                      420
dtype: int64
host_id                           int64
accommodates                      int64
beds               

In [34]:
# Lits of extra columns I want to check for missing values and types.

extra = ["hosts_time_as_host_years", "hosts_time_as_host_months",
         "minimum_minimum_nights", "minimum_nights_avg_ntm",
         "availability_30", "reviews_per_month",
         "host_total_listings_count", "calculated_host_listings_count"]

print(df[extra].isna().sum()) #how many blanks does each of those columns have
print(df["bathrooms_text"].value_counts().head(10)) #what values does the bathroom text have
print(df["host_is_superhost"].value_counts(dropna=False)) #what values does the superhost column have
print(df["property_type"].value_counts().head(8)) #what are the most common property types

hosts_time_as_host_years             0
hosts_time_as_host_months            0
minimum_minimum_nights             303
minimum_nights_avg_ntm               0
availability_30                      0
reviews_per_month                  420
host_total_listings_count         3591
calculated_host_listings_count       0
dtype: int64
bathrooms_text
1 bath       1875
2 baths      1450
1.5 baths     109
0 baths        74
2.5 baths      52
3 baths        27
4 baths         2
Half-bath       1
3.5 baths       1
Name: count, dtype: int64
host_is_superhost
f    2496
t    1095
Name: count, dtype: int64
property_type
Entire rental unit           3301
Entire condo                   92
Entire home                    69
Entire serviced apartment      66
Entire loft                    33
Room in aparthotel              6
Entire townhouse                5
Entire guest suite              4
Name: count, dtype: int64


In [35]:
# Clean data for the model to use
# BATHROOMS: turn text like "1.5 baths" into a number
df["bathrooms_num"] = (
    df["bathrooms_text"].str.lower()                         # make everything lowercase so matching is easy
    .str.replace("half-bath", "0.5", regex=False)            # "Half-bath" has no digit, so write it as 0.5
    .str.extract(r"(\d+\.?\d*)")[0]                          # pull out the first number in the text (e.g. 1.5)
    .astype(float)                                           # convert that text number into a real number
)

#  turn "t"/"f" into 1/0 so the model can use it 
df["is_superhost"] = (df["host_is_superhost"] == "t").astype(int)   

# empty cols these had zero values in this snapshot, so they are useless
empty_cols = ["bathrooms", "host_since", "host_response_rate",
              "host_acceptance_rate", "instant_bookable",
              "host_total_listings_count"]
df = df.drop(columns=empty_cols)                             # remove them from the table

# counts of each bathroom value
print(df["bathrooms_num"].value_counts(dropna=False))

bathrooms_num
1.0    1875
2.0    1450
1.5     109
0.0      74
2.5      52
3.0      27
4.0       2
0.5       1
3.5       1
Name: count, dtype: int64


Output explanation: there are no blanks and half bath is 0.5. Some apartments have no bathrooms because they are rooms rented out. 

In [36]:
# count how many listings there are of each property type (most common first)
print(df["property_type"].value_counts())

property_type
Entire rental unit           3301
Entire condo                   92
Entire home                    69
Entire serviced apartment      66
Entire loft                    33
Room in aparthotel              6
Entire townhouse                5
Entire guest suite              4
Entire guesthouse               4
Tiny home                       2
Entire place                    2
Entire vacation home            2
Casa particular                 2
Entire chalet                   2
Camper/RV                       1
Name: count, dtype: int64


#### Engineer features

In [37]:
import json         
import numpy as np   

# --- PROPERTY TYPE: 1 if it's the common "Entire rental unit", 0 for everything else ---
df["is_rental_unit"] = (df["property_type"] == "Entire rental unit").astype(int)

# --- AMENITIES: count how many amenities each listing has ---
# the column is text that looks like a list: ["Wifi", "Kitchen", ...]
# json.loads turns that text into a real list, len() counts the items
df["amenity_count"] = df["amenities"].apply(lambda text: len(json.loads(text)))

# --- DISTANCE TO CENTER: straight-line distance in km to Puerta del Sol (middle of Madrid) ---
center_lat, center_lon = 40.4168, -3.7038                    # coordinates of Puerta del Sol
lat1, lon1 = np.radians(df["latitude"]), np.radians(df["longitude"])   # convert degrees to radians
lat2, lon2 = np.radians(center_lat), np.radians(center_lon)
a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
df["dist_center_km"] = 6371 * 2 * np.arcsin(np.sqrt(a))      # haversine formula; 6371 = Earth's radius in km

# --- HAS REVIEWS: 1 if the listing has at least one review, 0 if it's new ---
df["has_reviews"] = (df["number_of_reviews"] > 0).astype(int)

# --- CHECK: look at the new columns ---
print(df[["is_rental_unit", "amenity_count", "dist_center_km", "has_reviews"]].describe())

       is_rental_unit  amenity_count  dist_center_km  has_reviews
count     3591.000000    3591.000000     3591.000000  3591.000000
mean         0.919243      30.230855        2.450327     0.883041
std          0.272500      11.681326        2.210993     0.321416
min          0.000000       3.000000        0.006050     0.000000
25%          1.000000      23.000000        0.784027     1.000000
50%          1.000000      30.000000        1.700350     1.000000
75%          1.000000      37.000000        3.569376     1.000000
max          1.000000      90.000000       13.488585     1.000000


Every column has 3,591 values, so there are no blanks in the new features.
is_rental_unit: the mean is 0.92, meaning 92% of listings are “Entire rental unit”. That matches the earlier count. This feature won’t separate many listings, which is worth a sentence in your report.
amenity_count: the typical listing has about 30 amenities, ranging from 3 to 90. That’s a plausible spread.
dist_center_km: the median is 1.7 km and 75% of listings are within 3.6 km, with the farthest at 13.5 km. That makes sense for a compact city like Madrid.
has_reviews: the mean is 0.883, so 88.3% have reviews and about 11.7% don’t. That matches the 420 blanks we saw (420 ÷ 3,591).

#### Splitting data 
All of host’s listings stay on the same side because  a new host’s flat is new to the model, so the test should look like that.

Train (70%): the model learns from it.
Validation (15%): the practice exam. You use it to tune the regularization strength and the learning rate.
Test (15%): the final exam. You look at it once, at the end.

In [38]:
from sklearn.model_selection import GroupShuffleSplit   # same splitter as before

targets = [0.70, 0.15, 0.15]          # the shares I WANT for train / validation / test

# --- TRY SEEDS 0 TO 9 (only looking at split sizes, never at model scores) ---
for seed in range(10):
    # first split: set aside the test hosts
    outer = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)
    tv_idx, te_idx = next(outer.split(df, groups=df["host_id"]))
    tv = df.iloc[tv_idx]
    # second split: take validation hosts out of the rest
    inner = GroupShuffleSplit(n_splits=1, test_size=0.176, random_state=seed)
    tr_idx, va_idx = next(inner.split(tv, groups=tv["host_id"]))

    sizes = [len(tr_idx), len(va_idx), len(te_idx)]          # listings in train / val / test
    shares = [round(s / len(df), 3) for s in sizes]          # as fractions of 3591
    worst_gap = max(abs(a - b) for a, b in zip(shares, targets))   # biggest miss vs the target
    print(seed, sizes, shares, "worst gap:", round(worst_gap, 3))   # smaller gap = better balanced

0 [2511, 625, 455] [0.699, 0.174, 0.127] worst gap: 0.024
1 [2475, 496, 620] [0.689, 0.138, 0.173] worst gap: 0.023
2 [2485, 475, 631] [0.692, 0.132, 0.176] worst gap: 0.026
3 [2598, 548, 445] [0.723, 0.153, 0.124] worst gap: 0.026
4 [2566, 527, 498] [0.715, 0.147, 0.139] worst gap: 0.015
5 [2567, 583, 441] [0.715, 0.162, 0.123] worst gap: 0.027
6 [2605, 574, 412] [0.725, 0.16, 0.115] worst gap: 0.035
7 [2532, 383, 676] [0.705, 0.107, 0.188] worst gap: 0.043
8 [2593, 445, 553] [0.722, 0.124, 0.154] worst gap: 0.026
9 [2296, 754, 541] [0.639, 0.21, 0.151] worst gap: 0.061


This code is because i want to pick teh seed with the best gap, instead of cherry pikcing. I split by host, so whole hosts move together, and a few hosts with many listings made my first split uneven (about 67% / 23% / 10%). A test set of only 367 listings gives a noisy final score. Before training any model, I compared seeds 0 to 9 and chose the one whose shares were closest to 70/15/15. I only looked at split sizes and never at model results, so the choice can’t favour my model. I tried 10 seeds in total.

I PIKCED SEED 4, WHICH HAD THE BEST GAP (0.015)

In [39]:
# --- FIRST SPLIT: set aside 15% of hosts as the TEST set ---
# groups=df["host_id"] means all listings from one host go to the same side
# random_state=4 makes the shuffle repeatable, so every run gives the same split
outer = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=4)
trainval_idx, test_idx = next(outer.split(df, groups=df["host_id"]))   # row positions for each side
trainval_df = df.iloc[trainval_idx]          # the other 85% (train + validation together)
test_df = df.iloc[test_idx].copy()           # the 15% test set

# --- SECOND SPLIT: from the 85%, take about 15% of the total as VALIDATION ---
# 0.176 of 85% is about 15% of the whole data set
inner = GroupShuffleSplit(n_splits=1, test_size=0.176, random_state=4)
train_idx, val_idx = next(inner.split(trainval_df, groups=trainval_df["host_id"]))   # row positions inside the 85%
train_df = trainval_df.iloc[train_idx].copy()   # the training set
val_df = trainval_df.iloc[val_idx].copy()       # the validation set

# --- CHECKS ---
print("listings:", len(train_df), len(val_df), len(test_df))   # should be 2566 527 498
print("hosts:", train_df["host_id"].nunique(), val_df["host_id"].nunique(), test_df["host_id"].nunique())
# hosts that appear on two sides at once: all three of these must print set()
print(set(train_df["host_id"]) & set(val_df["host_id"]) & set(test_df["host_id"]))
print(set(train_df["host_id"]) & set(test_df["host_id"]))
print(set(train_df["host_id"]) & set(val_df["host_id"]))

listings: 2566 527 498
hosts: 1211 259 260
set()
set()
set()


What I did: I split the data by host into three parts and got 2,566 listings in train, 527 in validation and 498 in test (about 71% / 15% / 14%).
Why by host: because a new host’s flat is new to the model, so the test should look like that. If one host’s similar flats were on both sides, the model could recognize them and the score would look better than it really is.
What the output showed:
The three parts add up to 3,591, so no listing was lost or counted twice.
There are 1,211 hosts in train, 259 in validation and 260 in test.
The three empty set() lines show that no host appears in two parts, so the split by host worked.
Seed choice: I tried seeds 0 to 9 plus 42 (11 splits). I chose seed 4 because its shares were closest to 70/15/15 (worst gap 0.015). I chose it by split balance only and never looked at model scores.

#### 5. Fill missing values (using training data only)

In [40]:
# --- WHICH COLUMNS GET THE "TYPICAL VALUE" (MEDIAN) FILL ---
median_cols = ["beds", "review_scores_rating", "review_scores_location"]

# --- CALCULATE the typical value from the TRAINING rows only ---
# (if we used all rows, the test set would quietly influence the number)
medians = train_df[median_cols].median()
print(medians)                                   # look at the values we will fill with

# --- APPLY the same fill to train, validation and test ---
for part in [train_df, val_df, test_df]:
    part[median_cols] = part[median_cols].fillna(medians)             # blanks -> the training median
    part["reviews_per_month"] = part["reviews_per_month"].fillna(0)   # no reviews yet -> 0 reviews per month

# --- CHECK: every one of these should now show 0 blanks ---
check_cols = median_cols + ["reviews_per_month"]
print(train_df[check_cols].isna().sum())
print(val_df[check_cols].isna().sum())
print(test_df[check_cols].isna().sum())

beds                      3.00
review_scores_rating      4.76
review_scores_location    4.86
dtype: float64
beds                      0
review_scores_rating      0
review_scores_location    0
reviews_per_month         0
dtype: int64
beds                      0
review_scores_rating      0
review_scores_location    0
reviews_per_month         0
dtype: int64
beds                      0
review_scores_rating      0
review_scores_location    0
reviews_per_month         0
dtype: int64


Medians from the training set: beds 3.00, review score rating 4.76, review score location 4.86. These are the numbers that replaced the blanks in all three parts.
Three tables of zeros: one each for train, validation and test, so every column you checked has no blanks left.
Beds of 3.00 for a 2-bedroom flat: that’s plausible, since 2-bedroom flats usually have 3 beds.
Review scores near 4.8: Airbnb scores are almost always high, so they won’t separate listings much. Worth a sentence in your report.
What the notes for this step should say

-> I filled blanks in beds and the review scores with the training median, and reviews_per_month with 0 because a blank means no reviews yet.
I calculated the medians on the training set only, so the test set can’t influence the numbers.
All three parts now have 0 blanks in those columns.

In [41]:
#Before deciding what features to choose, i will measures how strongly each numeric candidate moves with price
# --- NUMERIC CANDIDATE FEATURES I WANT TO COMPARE WITH PRICE ---
num_candidates = ["accommodates", "beds", "bathrooms_num", "amenity_count", "dist_center_km",
                  "is_rental_unit", "is_superhost", "calculated_host_listings_count",
                  "hosts_time_as_host_years", "minimum_nights_avg_ntm", "availability_365",
                  "number_of_reviews", "reviews_per_month", "review_scores_rating",
                  "review_scores_location", "has_reviews"]

# --- CORRELATION WITH PRICE (training rows only) ---
# +1 = rises together with price, -1 = moves the opposite way, 0 = no straight-line link
corr = train_df[num_candidates + ["price"]].corr()["price"].drop("price")   # correlation of each column with price
print(corr.sort_values(ascending=False).round(2))                           # strongest positive first

accommodates                      0.25
bathrooms_num                     0.23
beds                              0.19
review_scores_location            0.19
reviews_per_month                 0.17
number_of_reviews                 0.16
amenity_count                     0.15
has_reviews                       0.15
review_scores_rating              0.10
hosts_time_as_host_years          0.10
is_superhost                      0.10
is_rental_unit                    0.05
availability_365                 -0.04
calculated_host_listings_count   -0.10
dist_center_km                   -0.28
minimum_nights_avg_ntm           -0.34
Name: price, dtype: float64


- minimum_nights_avg_ntm (-0.34): the strongest link. Listings that demand longer minimum stays tend to have a lower nightly price. My guess is that long-stay and monthly rentals are cheaper per night, but that’s a hypothesis.
- dist_center_km (-0.28): the farther from the center, the cheaper, this is second strongest.
- accommodates (0.25), bathrooms_num (0.23), beds (0.19): bigger flats cost more, as expected. These three overlap, since a flat that sleeps more has more beds and bathrooms.
Review columns (0.10 to 0.19): small effects. number_of_reviews, reviews_per_month and has_reviews measure similar things.
availability_365 (-0.04) and is_rental_unit (0.05): almost no straight-line link.

IMPORTANT LIMITATION: Because the links are weak, linear regression will probably explain only part of the price. That’s normal for Airbnb data, since things like photos, view and building quality aren’t in your columns.

I measured how strongly each numeric column moves with price, using the training set only. All the links were weak, between -0.34 and +0.25. The strongest were minimum nights (-0.34), distance to the center (-0.28) and accommodates (0.25). I decided to follow the numbers: I keep any column with a correlation of at least 0.10 in either direction, and drop the rest. Some kept columns overlap (accommodates, beds and bathrooms), so the model will share the credit between them. Because the links are weak, linear regression will only explain part of the price, since things like photos and building quality aren’t in the data.

#### Choose features and build model tables

In [42]:
# --- RULE: keep a numeric column if its correlation with price is at least 0.10 (up or down) ---
# corr was calculated on the TRAINING rows only, so no test information is used
chosen_numeric = [c for c in corr.index if abs(corr[c]) >= 0.10]
dropped = [c for c in corr.index if abs(corr[c]) < 0.10]
print("keeping:", chosen_numeric)          # the columns that passed the rule
print("dropping:", dropped)                # the columns that did not

# --- DISTRICT: turn the text column into one yes/no column per district ---
# the district list is learned from the TRAINING rows only
# drop_first=True removes one district, which avoids a redundant column for linear regression
district_train = pd.get_dummies(train_df["neighbourhood_group_cleansed"], prefix="district", drop_first=True)
district_cols = list(district_train.columns)         # the district columns the model will use

# --- BUILD ONE TABLE OF INPUTS (X) AND ONE LIST OF ANSWERS (y) FOR EACH PART ---
def build_xy(part):
    district = pd.get_dummies(part["neighbourhood_group_cleansed"], prefix="district")
    district = district.reindex(columns=district_cols, fill_value=0)   # same columns as training; missing district -> 0
    X = pd.concat([part[chosen_numeric], district], axis=1).astype(float)   # numbers + district columns
    y = part["price"].astype(float)                                         # the price we want to predict
    return X, y

X_train, y_train = build_xy(train_df)    # training inputs and answers
X_val, y_val = build_xy(val_df)          # validation inputs and answers
X_test, y_test = build_xy(test_df)       # test inputs and answers

# --- CHECKS ---
print(X_train.shape, X_val.shape, X_test.shape)               # same number of columns in all three
print(list(X_train.columns) == list(X_val.columns) == list(X_test.columns))   # must print True
print(X_train.isna().sum().sum(), X_val.isna().sum().sum(), X_test.isna().sum().sum())   # must print 0 0 0

keeping: ['accommodates', 'beds', 'bathrooms_num', 'amenity_count', 'dist_center_km', 'calculated_host_listings_count', 'minimum_nights_avg_ntm', 'number_of_reviews', 'reviews_per_month', 'review_scores_rating', 'review_scores_location', 'has_reviews']
dropping: ['is_rental_unit', 'is_superhost', 'hosts_time_as_host_years', 'availability_365']
(2566, 32) (527, 32) (498, 32)
True
0 0 0


The rule dropped four columns: is_rental_unit, availability_365, is_superhost and hosts_time_as_host_years. The last two showed 0.10 in your table, but that was rounded. Their real values must be slightly below 0.10, so the rule cut them. That’s the weakness I mentioned: a cutoff of 0.10 is arbitrary, and columns right at the edge fall on either side by tiny amounts.

Don’t change the rule now just because of this, because that would be choosing features after seeing results. In your report, say that the cutoff is arbitrary and that two borderline columns (superhost and host experience) were dropped by a tiny margin. If you want to override the rule, tell me and give your reason.

#### Scaling numbers

In [43]:

# --- WHY SCALE: amenity_count is around 30 but has_reviews is 0 or 1 ---
# the PyTorch versions learn in small steps (gradient descent), and columns on very different
# scales make that slow and unstable, so every number column is rescaled as (value - mean) / spread

scaler = StandardScaler()                      # the tool that does the rescaling
scaler.fit(X_train[chosen_numeric])            # learn the mean and spread from the TRAINING rows only

# --- APPLY the same rescaling to train, validation and test ---
X_train_s = X_train.copy()                     # copies, so the unscaled tables are kept
X_val_s = X_val.copy()
X_test_s = X_test.copy()
for X_s in [X_train_s, X_val_s, X_test_s]:
    X_s[chosen_numeric] = scaler.transform(X_s[chosen_numeric])   # district yes/no columns stay 0/1

# --- CHECK: training columns should have mean about 0 and spread about 1 ---
print(X_train_s[chosen_numeric].mean().round(2).max(), X_train_s[chosen_numeric].std().round(2).max())
# validation and test will NOT be exactly 0 and 1, and that is expected
print(X_val_s[chosen_numeric].mean().round(2).abs().max(), X_test_s[chosen_numeric].mean().round(2).abs().max())

0.0 1.0
0.42 0.39


The first line should print something like 0.0 1.0, because training columns now have a mean of about 0 and a spread of about 1.
The second line prints the biggest average in validation and test. It should be small, like 0.1 to 0.2, but not exactly 0. That’s fine, because those parts were scaled with the training numbers.
Why training only, again

As with the medians, the mean and spread come from the training rows. If you used all rows, the test set would quietly influence the scaling.

0.0 1.0: the training columns now have a mean of about 0 and a spread of about 1. That’s what scaling should do.
0.42 0.39: the biggest average in validation and in test. These are measured in training spreads, so some column’s average in those parts sits about 0.4 spreads away from the training average, this howver is double than what expected. It isn’t an error, and the scaling is correct. It means at least one column looks different in validation and test than in training.

I guess its because we split by host, so whole hosts moved together. A column like calculated_host_listings_count (how many listings a host has) or number_of_reviews has a long tail of big operators and very popular flats. If a few of those landed in one part, that part’s average shifts. 

If validation and test look different from training, a score measured on them can be better or worse than on future flats, simply because of which hosts landed where. That’s a limitation of a host split with a single seed. I SAID IT BEFORE: Splitting by host is a harder test than a random split, because the model meets hosts it has never seen.

So i will check which column it is below. 

In [44]:
# --- WHICH COLUMNS DIFFER MOST BETWEEN TRAIN AND VALIDATION/TEST? ---
# after scaling, the training mean of every column is 0, so these numbers show the gap in "training spreads"
shift = pd.DataFrame({
    "val_mean": X_val_s[chosen_numeric].mean(),    # average of each column in the validation set
    "test_mean": X_test_s[chosen_numeric].mean(),  # average of each column in the test set
}).round(2)
print(shift.sort_values("val_mean", key=abs, ascending=False))   # biggest gap first

                                val_mean  test_mean
calculated_host_listings_count     -0.42      -0.39
reviews_per_month                   0.32       0.19
has_reviews                         0.24       0.22
number_of_reviews                   0.20       0.04
amenity_count                       0.16       0.02
review_scores_location              0.13      -0.06
minimum_nights_avg_ntm             -0.12      -0.20
accommodates                        0.09       0.12
review_scores_rating                0.08      -0.02
dist_center_km                     -0.07       0.15
bathrooms_num                       0.06      -0.01
beds                                0.04      -0.03


The split is by host with one seed, so the three parts differ a bit. The biggest gap is the host’s number of listings (about -0.4 training spreads in validation and test), because large operators ended up mostly in training. Scores on validation and test could be a little better or worse than on future flats for that reason.

## Modeling: three implementations, one model

I predict nightly price, so the model is linear regression. I train the same model three ways on the same split and the same scaled inputs:
1. scikit-learn
2. a manual PyTorch loop (raw tensors, autograd, hand-written update step)
3. the standard PyTorch workflow (nn.Module + torch.optim)

I compare all three with each other and with a naive baseline that always guesses the median training price. Scores are in euros (MAE), because that's easy to explain to a host. I only look at train and validation here, and the test set is used once at the very end.

In [45]:
# Baseline and method 1
from sklearn.metrics import mean_absolute_error, r2_score    # MAE = average miss in euros, R2 = share of price variation explained

results = {}    # an empty table; each method adds its scores here so we can compare them later

# --- NAIVE BASELINE: ignore all features, always guess the median training price ---
baseline_price = y_train.median()                                                # the one price the baseline always guesses
base_train_pred = [baseline_price] * len(y_train)                                # the same guess for every training flat
base_val_pred = [baseline_price] * len(y_val)                                    # the same guess for every validation flat
results["baseline"] = {
    "train_MAE": mean_absolute_error(y_train, base_train_pred),                  # average miss on training flats
    "val_MAE": mean_absolute_error(y_val, base_val_pred),                        # average miss on validation flats
    "val_R2": r2_score(y_val, base_val_pred),                                    # will be about 0 or slightly negative
}

# --- METHOD 1: SCIKIT-LEARN LINEAR REGRESSION on the scaled inputs ---
sk_model = LinearRegression()          # the model: price = weights x features + a constant
sk_model.fit(X_train_s, y_train)       # learn the weights from the TRAINING rows only
sk_train_pred = sk_model.predict(X_train_s)    # predicted prices for training flats
sk_val_pred = sk_model.predict(X_val_s)        # predicted prices for validation flats
results["sklearn"] = {
    "train_MAE": mean_absolute_error(y_train, sk_train_pred),
    "val_MAE": mean_absolute_error(y_val, sk_val_pred),
    "val_R2": r2_score(y_val, sk_val_pred),
}

# --- SHOW THE RESULTS TABLE (rounded) ---
print(pd.DataFrame(results).T.round(2))        # one row per method, one column per score

          train_MAE  val_MAE  val_R2
baseline      73.78    69.89   -0.07
sklearn       60.88    58.67    0.27


The model beats the baseline. Guessing the median price misses by about €70 on validation flats, and the model misses by about €59. That’s roughly €11 better, or 16%. It’s a modest gain, which fits the weak correlations.
No overfitting. Train and validation errors are close (60.88 vs 58.67). Validation is slightly better than train, which is possible when the validation hosts happen to be a bit easier to price.
R² of 0.27 means the model explains about 27% of the price variation. A baseline R² of -0.07 just means the median guess was slightly worse than the average for that measure. Both go in your limitations: most of the price differences between flats aren’t captured by your columns.

In [46]:
# Manual pytorch loop

import torch                                   # PyTorch: tensors + automatic gradients (autograd)

# --- TURN THE TABLES INTO TENSORS (PyTorch's version of arrays) ---
X_tr = torch.tensor(X_train_s.values, dtype=torch.float32)                  # training inputs, shape (2566, 32)
y_tr = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)     # training prices as one column
X_va = torch.tensor(X_val_s.values, dtype=torch.float32)                    # validation inputs
y_va = torch.tensor(y_val.values, dtype=torch.float32).reshape(-1, 1)       # validation prices

# --- THE MODEL'S PARAMETERS: one weight per feature plus one constant (bias), starting at 0 ---
n_features = X_tr.shape[1]                                          # 32
w = torch.zeros(n_features, 1, requires_grad=True)                  # requires_grad=True: autograd tracks these
b = torch.zeros(1, requires_grad=True)

lr = 0.05          # learning rate: how big each update step is
epochs = 5000      # how many times we go through the full training set

# --- THE TRAINING LOOP ---
for epoch in range(epochs):
    pred = X_tr @ w + b                          # 1. predict: price = features x weights + bias
    loss = ((pred - y_tr) ** 2).mean()           # 2. loss: mean squared error (how wrong, on average)
    loss.backward()                              # 3. autograd computes the gradient of the loss for w and b
    with torch.no_grad():                        # 4. update by hand, without tracking this step
        w -= lr * w.grad                         #    move each weight a small step against its gradient
        b -= lr * b.grad                         #    same for the bias
    w.grad.zero_()                               # 5. clear the gradients, or they pile up
    b.grad.zero_()
    if epoch % 1000 == 0:
        print(epoch, round(loss.item(), 1))      # the loss should go down

# --- PREDICT AND SCORE (train and validation) ---
with torch.no_grad():                            # no gradients needed for predictions
    man_train_pred = (X_tr @ w + b).numpy().ravel()    # back to plain numbers
    man_val_pred = (X_va @ w + b).numpy().ravel()
results["manual_torch"] = {
    "train_MAE": mean_absolute_error(y_train, man_train_pred),
    "val_MAE": mean_absolute_error(y_val, man_val_pred),
    "val_R2": r2_score(y_val, man_val_pred),
}

# --- DO THE MANUAL WEIGHTS MATCH SCIKIT-LEARN'S WEIGHTS? ---
print("biggest weight difference vs sklearn:", float(abs(w.detach().numpy().ravel() - sk_model.coef_).max()))
print(pd.DataFrame(results).T.round(2))

0 55105.0
1000 6168.1
2000 6161.4
3000 6159.1
4000 6158.1
biggest weight difference vs sklearn: 19.171992569677663
              train_MAE  val_MAE  val_R2
baseline          73.78    69.89   -0.07
sklearn           60.88    58.67    0.27
manual_torch      60.91    58.70    0.27


The manual loop works. The predictions match scikit-learn almost exactly, but the weights don’t, and that gap is worth investigating.
The biggest weight difference is 19.17, which is not small. Weights are in euros per scaled unit, so some weight differs by about €19 between the two methods, even though the predictions are the same.

Maybe its because columns overlap: When columns overlap, many weight combinations give nearly the same predictions. scikit-learn solves it exactly, and gradient descent slides slowly along that flat valley and stops a bit short.

In [47]:
# --- LINE UP THE WEIGHTS FROM BOTH METHODS, ONE ROW PER FEATURE ---
compare_w = pd.DataFrame({
    "sklearn": sk_model.coef_,                          # weights learned by scikit-learn
    "manual": w.detach().numpy().ravel(),               # weights learned by my manual loop
}, index=X_train_s.columns)                             # label each row with its feature name
compare_w["diff"] = (compare_w["manual"] - compare_w["sklearn"]).abs()   # how far apart the two are
print(compare_w.sort_values("diff", ascending=False).head(8).round(2))   # the 8 biggest gaps

# --- THE CONSTANT TERM (the price when every scaled feature is 0) ---
print("sklearn intercept:", round(float(sk_model.intercept_), 2), " manual bias:", round(float(b.item()), 2))

# --- HOW MANY TRAINING FLATS FALL IN EACH DISTRICT? (few flats = shaky weight) ---
print(X_train_s[district_cols].sum().sort_values().head(8))

                                sklearn     manual   diff
district_Barajas                  48.42  29.250000  19.17
district_Villa de Vallecas        25.98   9.070000  16.90
district_San Blas - Canillejas    28.90  18.719999  10.18
district_Vicálvaro                 5.40  -4.360000   9.76
district_Hortaleza                33.94  25.610001   8.33
district_Villaverde                1.73  -6.210000   7.94
district_Fuencarral - El Pardo    27.95  20.010000   7.94
district_Ciudad Lineal             9.14   4.630000   4.51
sklearn intercept: 194.51  manual bias: 195.73
district_Villa de Vallecas         5.0
district_Vicálvaro                 9.0
district_Barajas                  11.0
district_Moratalaz                11.0
district_Villaverde               12.0
district_San Blas - Canillejas    26.0
district_Fuencarral - El Pardo    27.0
district_Hortaleza                33.0
dtype: float64


The gaps line up with how few flats each district has.Why it makes sense: with only 5 to 12 flats, the data barely pins down those weights, so the loss surface is nearly flat along them. scikit-learn solves it exactly. The manual loop shrinks small, weakly supported weights toward 0 during its slow descent and stops before it fully moves them. That’s my interpretation of the pattern, so say “likely” in your report.the predictions and scores agree within a few cents. The weights differ in the rarest districts (5 to 12 training flats), which are weakly determined, and the manual loop hasn’t fully converged on them. The gap doesn’t change the predictions in any meaningful way.

I will tune this later!

In [48]:
# Standard pytorch (nn.Module + torch.optim)

import torch.nn as nn                                # nn: PyTorch's building blocks for models

# --- THE MODEL AS A CLASS: one linear layer = weights x features + bias ---
class LinearReg(nn.Module):                          # nn.Module is the standard base class for models
    def __init__(self, n_inputs):
        super().__init__()                           # set up the parent class
        self.linear = nn.Linear(n_inputs, 1)         # one weight per input plus a bias, 1 output (price)
    def forward(self, x):                            # what happens when data goes through the model
        return self.linear(x)

torch.manual_seed(0)                                 # makes the random starting values repeatable
nn_model = LinearReg(n_features)                     # create the model (n_features = 32, from the manual loop)
nn.init.zeros_(nn_model.linear.weight)               # start the weights at 0, same as the manual loop
nn.init.zeros_(nn_model.linear.bias)                 # start the bias at 0 too, so the comparison is fair

loss_fn = nn.MSELoss()                                                   # same loss as before: mean squared error
optimizer = torch.optim.SGD(nn_model.parameters(), lr=0.05)              # plain gradient descent, same learning rate

# --- THE TRAINING LOOP: the optimizer replaces my hand-written update step ---
for epoch in range(5000):                            # same number of epochs as the manual loop
    pred = nn_model(X_tr)                            # 1. predict
    loss = loss_fn(pred, y_tr)                       # 2. loss
    optimizer.zero_grad()                            # 3. clear old gradients
    loss.backward()                                  # 4. autograd computes the gradients
    optimizer.step()                                 # 5. the optimizer updates the weights
    if epoch % 1000 == 0:
        print(epoch, round(loss.item(), 1))          # the loss should go down

# --- PREDICT AND SCORE ---
with torch.no_grad():
    nn_train_pred = nn_model(X_tr).numpy().ravel()
    nn_val_pred = nn_model(X_va).numpy().ravel()
results["nn_torch"] = {
    "train_MAE": mean_absolute_error(y_train, nn_train_pred),
    "val_MAE": mean_absolute_error(y_val, nn_val_pred),
    "val_R2": r2_score(y_val, nn_val_pred),
}
print(pd.DataFrame(results).T.round(2))

0 55105.0
1000 6168.1
2000 6161.4
3000 6159.1
4000 6158.1
              train_MAE  val_MAE  val_R2
baseline          73.78    69.89   -0.07
sklearn           60.88    58.67    0.27
manual_torch      60.91    58.70    0.27
nn_torch          60.91    58.70    0.27


All three methods are trained, and the results table is clean.

All three methods give the same predictions to within about 3 cents (validation MAE 58.67 / 58.70 / 58.70, R² 0.27). Weights differ by up to about €19 in the rarest districts (5 to 12 training flats), which are weakly determined, and the gradient-descent versions haven’t fully converged on them.

#### Tuning 
"Only two things are worth touching: regularization strength and the PyTorch learning rate." Regularization shrinks big weights, which could help with the shaky district weights. Choosing is done on the validation set, and the test set stays untouched.

In [49]:
#### regularisation strength
from sklearn.linear_model import Ridge        # linear regression with a penalty on big weights (alpha = penalty strength)

# --- TRY A FEW PENALTY STRENGTHS (alpha); a bigger alpha shrinks the weights more ---
alphas = [0.1, 1, 10, 100, 1000]               # a small grid on purpose, because tuning should be light
tune_rows = []
for a in alphas:
    ridge = Ridge(alpha=a)                     # the same model as before, plus the penalty
    ridge.fit(X_train_s, y_train)              # learn from the TRAINING rows only
    tune_rows.append({
        "alpha": a,
        "train_MAE": mean_absolute_error(y_train, ridge.predict(X_train_s)),
        "val_MAE": mean_absolute_error(y_val, ridge.predict(X_val_s)),   # the score we choose by
    })

# --- SHOW THE RESULTS NEXT TO THE PLAIN MODEL (no penalty, val MAE 58.67) ---
print(pd.DataFrame(tune_rows).round(2))
print("plain linear regression val MAE:", round(results["sklearn"]["val_MAE"], 2))

    alpha  train_MAE  val_MAE
0     0.1      60.88    58.67
1     1.0      60.90    58.68
2    10.0      60.99    58.79
3   100.0      61.56    59.36
4  1000.0      63.14    61.00
plain linear regression val MAE: 58.67


Regularization doesn’t help, so you keep plain linear regression.I tried 5 penalty strengths (0.1 to 1000) and none beat plain linear regression on validation, so I kept it without regularization.Finding that a tuning step didn’t help is a fine result.

In [50]:
# Learning rate 

# --- A SMALL FUNCTION THAT TRAINS THE PYTORCH MODEL WITH A GIVEN LEARNING RATE ---
def train_torch(lr, epochs=5000):
    torch.manual_seed(0)                                   # same random start every time, so runs are comparable
    model = LinearReg(n_features)                          # the same model class as method 3
    nn.init.zeros_(model.linear.weight)                    # start the weights at 0
    nn.init.zeros_(model.linear.bias)                      # start the bias at 0
    opt = torch.optim.SGD(model.parameters(), lr=lr)       # plain gradient descent with this learning rate
    for _ in range(epochs):                                # repeat the training step
        loss = nn.MSELoss()(model(X_tr), y_tr)             # predict and measure the loss
        opt.zero_grad()                                    # clear old gradients
        loss.backward()                                    # compute new gradients
        opt.step()                                         # update the weights
    return model

# --- TRY A FEW LEARNING RATES AND COMPARE THEM TO SCIKIT-LEARN ---
lr_rows = []
for lr in [0.05, 0.1, 0.2]:                                # a small grid on purpose
    m = train_torch(lr)                                    # train with this learning rate
    with torch.no_grad():                                  # no gradients needed to predict
        val_pred = m(X_va).numpy().ravel()                 # predicted validation prices
    weight_gap = abs(m.linear.weight.detach().numpy().ravel() - sk_model.coef_).max()   # biggest weight gap vs sklearn
    lr_rows.append({"lr": lr,
                    "val_MAE": mean_absolute_error(y_val, val_pred),   # the score we choose by (nan = it blew up)
                    "weight_gap_vs_sklearn": weight_gap})
print(pd.DataFrame(lr_rows).round(3))
print("sklearn val MAE:", round(results["sklearn"]["val_MAE"], 3))   # the target to get close to

     lr  val_MAE  weight_gap_vs_sklearn
0  0.05   58.700                 19.172
1  0.10   58.669                 10.837
2  0.20   58.668                  4.868
sklearn val MAE: 58.672


### Final models and test score

In [51]:
final_lr = 0.2                                                                # the learning rate chosen on validation
X_te = torch.tensor(X_test_s.values, dtype=torch.float32)                     # test inputs as a tensor

# --- A MANUAL-LOOP FUNCTION (the same loop as method 2, with a chosen learning rate) ---
def train_manual(lr, epochs=5000):
    w_ = torch.zeros(n_features, 1, requires_grad=True)       # weights start at 0
    b_ = torch.zeros(1, requires_grad=True)                   # bias starts at 0
    for _ in range(epochs):
        loss = (((X_tr @ w_ + b_) - y_tr) ** 2).mean()        # predict, then mean squared error
        loss.backward()                                       # autograd computes the gradients
        with torch.no_grad():
            w_ -= lr * w_.grad                                # hand-written update step
            b_ -= lr * b_.grad
        w_.grad.zero_()                                       # clear gradients
        b_.grad.zero_()
    return w_.detach(), b_.detach()

# --- TRAIN THE FINAL PYTORCH MODELS (training data only) ---
final_w, final_b = train_manual(final_lr)                     # method 2, final version
nn_final = train_torch(final_lr)                              # method 3, final version (function from the last cell)

# --- PREDICTIONS FOR TRAIN / VALIDATION / TEST FROM EACH METHOD ---
with torch.no_grad():
    preds = {
        "baseline": [[baseline_price] * len(y) for y in (y_train, y_val, y_test)],
        "sklearn": [sk_model.predict(X) for X in (X_train_s, X_val_s, X_test_s)],
        "manual_torch": [(X @ final_w + final_b).numpy().ravel() for X in (X_tr, X_va, X_te)],
        "nn_torch": [nn_final(X).numpy().ravel() for X in (X_tr, X_va, X_te)],
    }

# --- THE FINAL TABLE: the test set is looked at HERE, once, and not used to change anything ---
final_results = {}
for name, (p_tr, p_va, p_te) in preds.items():
    final_results[name] = {
        "train_MAE": mean_absolute_error(y_train, p_tr),
        "val_MAE": mean_absolute_error(y_val, p_va),
        "test_MAE": mean_absolute_error(y_test, p_te),
        "test_R2": r2_score(y_test, p_te),
    }
print(pd.DataFrame(final_results).T.round(2))

# --- DO THE THREE METHODS AGREE? (largest gap between any two predictions on the test set) ---
print("sklearn vs manual (max price gap, euros):", float(abs(preds["sklearn"][2] - preds["manual_torch"][2]).max()))
print("manual vs nn      (max price gap, euros):", float(abs(preds["manual_torch"][2] - preds["nn_torch"][2]).max()))

              train_MAE  val_MAE  test_MAE  test_R2
baseline          73.78    69.89     70.88    -0.02
sklearn           60.88    58.67     62.09     0.29
manual_torch      60.89    58.67     62.09     0.29
nn_torch          60.89    58.67     62.09     0.29
sklearn vs manual (max price gap, euros): 2.387628851245296
manual vs nn      (max price gap, euros): 3.0517578125e-05


After you see the test numbers, don’t go back and change anything (features, learning rate, cutoff) to improve them. Doing that would turn the test set into a second validation set, and the final score would no longer be honest. Whatever the numbers are, they go in the report as they are.

All three methods reach the same test MAE (about €62.09) and R² (0.29), against a baseline of €70.88. The two PyTorch versions are identical to within 0.00003 euros. scikit-learn differs from them by at most €2.39 on a single flat, probably because of the rare-district weights, where gradient descent hasn’t fully converged.

### Saving eveyrthing (for app) 

In [52]:
import os, json, joblib                                  # os = folders, json = text files, joblib = saves scikit-learn objects

os.makedirs("artifacts", exist_ok=True)                  # create the folder "artifacts" if it doesn't exist yet

# --- THE THREE MODELS ---
joblib.dump(sk_model, "artifacts/sklearn_model.joblib")                    # method 1: scikit-learn model
torch.save({"w": final_w, "b": final_b}, "artifacts/manual_torch.pt")      # method 2: the weights from the manual loop
torch.save(nn_final.state_dict(), "artifacts/nn_torch.pt")                 # method 3: the weights of the nn.Module model

# --- EVERYTHING NEEDED TO TURN A NEW FLAT INTO THE SAME NUMBERS ---
joblib.dump(scaler, "artifacts/scaler.joblib")                             # the mean and spread learned from training
meta = {
    "chosen_numeric": chosen_numeric,                    # the numeric feature names
    "district_cols": district_cols,                      # the district columns
    "feature_order": list(X_train_s.columns),            # the exact column order the models expect
    "medians": medians.to_dict(),                        # fill values for blanks (from training only)
    "baseline_price": float(baseline_price),             # the naive baseline's guess
    "center": [center_lat, center_lon],                  # Puerta del Sol, for the distance feature
    "final_lr": final_lr,                                # the chosen learning rate
    "n_features": int(n_features),                       # number of input columns
    "price_cap": 500,                                    # the price cutoff used in the data scope
}
with open("artifacts/meta.json", "w") as f:              # save as a readable text file
    json.dump(meta, f, indent=2)

# --- RESULTS TABLE (for the dashboard's comparison view) ---
pd.DataFrame(final_results).T.to_csv("artifacts/results.csv")              # one row per method

# --- ONE TABLE OF EVERY FLAT: unscaled features, true price, and each method's prediction ---
parts = []
for split, y_part, X_raw, i in [("train", y_train, X_train, 0), ("val", y_val, X_val, 1), ("test", y_test, X_test, 2)]:
    t = X_raw.copy()                                     # the unscaled features, easy to plot and read
    t["split"] = split                                   # which part this flat belongs to
    t["price"] = y_part.values                           # the true price
    for name in preds:
        t["pred_" + name] = np.asarray(preds[name][i]).ravel()   # that method's predicted price
    parts.append(t)
pd.concat(parts).to_csv("artifacts/predictions.csv", index=False)

# --- CHECK: do the saved files give back the same predictions? ---
sk_loaded = joblib.load("artifacts/sklearn_model.joblib")                      # load method 1 from disk
print("sklearn   :", float(abs(sk_loaded.predict(X_test_s) - preds["sklearn"][2]).max()))

saved = torch.load("artifacts/manual_torch.pt")                                # load method 2 from disk
print("manual    :", float(abs((X_te @ saved["w"] + saved["b"]).numpy().ravel() - preds["manual_torch"][2]).max()))

nn_loaded = LinearReg(n_features)                                              # empty model of the same shape
nn_loaded.load_state_dict(torch.load("artifacts/nn_torch.pt"))                  # fill it with the saved weights
with torch.no_grad():
    print("nn_torch  :", float(abs(nn_loaded(X_te).numpy().ravel() - preds["nn_torch"][2]).max()))
print(os.listdir("artifacts"))                                                 # list the saved files

sklearn   : 0.0
manual    : 0.0
nn_torch  : 0.0
['nn_torch.pt', 'manual_torch.pt', 'predictions.csv', 'sklearn_model.joblib', 'scaler.joblib', 'results.csv', 'meta.json']
